<a href="https://colab.research.google.com/github/kangyeson/AI-Class/blob/main/1012/abalone_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# 데이터 로드
url = "/content/drive/MyDrive/Colab Notebooks/abalone.csv"

data = pd.read_csv(url)

print(data.head())
print(data.shape)

   id Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  \
0   0   M   0.455     0.365   0.095        0.5140          0.2245   
1   1   M   0.350     0.265   0.090        0.2255          0.0995   
2   2   F   0.530     0.420   0.135        0.6770          0.2565   
3   3   M   0.440     0.365   0.125        0.5160          0.2155   
4   4   I   0.330     0.255   0.080        0.2050          0.0895   

   Viscera_weight  Shell_weight  Rings  
0          0.1010         0.150     15  
1          0.0485         0.070      7  
2          0.1415         0.210      9  
3          0.1140         0.155     10  
4          0.0395         0.055      7  
(4177, 10)


In [2]:
data = data.drop("id", axis=1)

# 범주형 데이터 원-핫 인코딩
data = pd.get_dummies(data, columns=["Sex"], dtype=int)

# 입력과 타겟 분리
X = data.drop("Rings", axis=1).values.astype(np.float32)
y = data["Rings"].values.astype(np.float32)

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [3]:
# 데이터 표준화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# TensorDataSet으로 랩 변환
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False
)

In [4]:
# 회귀 모델 정의
class RegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = RegressionModel(X_train.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [5]:
# 학습 루프
for epoch in range(50):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        output = model(X_batch)
        loss = criterion(output, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}, "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )

Epoch 1, Loss: 64.3342
Epoch 2, Loss: 13.3793
Epoch 3, Loss: 6.9520
Epoch 4, Loss: 6.0748
Epoch 5, Loss: 5.6894
Epoch 6, Loss: 5.4658
Epoch 7, Loss: 5.3389
Epoch 8, Loss: 5.1698
Epoch 9, Loss: 5.0398
Epoch 10, Loss: 5.0189
Epoch 11, Loss: 4.9226
Epoch 12, Loss: 4.9235
Epoch 13, Loss: 4.8732
Epoch 14, Loss: 4.7449
Epoch 15, Loss: 4.8129
Epoch 16, Loss: 4.7200
Epoch 17, Loss: 4.6671
Epoch 18, Loss: 4.6205
Epoch 19, Loss: 4.5919
Epoch 20, Loss: 4.5723
Epoch 21, Loss: 4.5730
Epoch 22, Loss: 4.5027
Epoch 23, Loss: 4.5239
Epoch 24, Loss: 4.4986
Epoch 25, Loss: 4.4543
Epoch 26, Loss: 4.4673
Epoch 27, Loss: 4.4920
Epoch 28, Loss: 4.4634
Epoch 29, Loss: 4.4241
Epoch 30, Loss: 4.4000
Epoch 31, Loss: 4.4446
Epoch 32, Loss: 4.4107
Epoch 33, Loss: 4.3549
Epoch 34, Loss: 4.3504
Epoch 35, Loss: 4.3645
Epoch 36, Loss: 4.3567
Epoch 37, Loss: 4.3200
Epoch 38, Loss: 4.3124
Epoch 39, Loss: 4.3046
Epoch 40, Loss: 4.3341
Epoch 41, Loss: 4.2922
Epoch 42, Loss: 4.2767
Epoch 43, Loss: 4.3045
Epoch 44, Loss: 4.

In [6]:
# 평가
model.eval()
preds, actuals = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch).cpu().numpy().flatten()

        preds.extend(outputs)
        actuals.extend(y_batch.numpy().flatten())

mse = mean_squared_error(actuals, preds)
print(f"Test MSE: {mse:.4f}")

Test MSE: 4.5555
